# 02 — Backend zoo: one factory, six backends

**Goal:** show how `BackendConfig` (Pydantic discriminated union) + `build_estimator(cfg)` (plain-dict registry dispatch) + `AbstractEstimator` (sklearn-compatible ABC) together let CV, HPO, SHAP, and ONNX export all run uniformly across TF MLP / Torch MLP / XGB / LGBM / CatBoost / FT-Transformer.


In [ ]:
import importlib.util

from deepCab.models import build_estimator
from deepCab.models._kinds import BACKENDS
from deepCab.schemas.config import (
    CatBoostConfig, FTTransformerConfig, LGBMConfig,
    TFMLPConfig, TorchMLPConfig, XGBConfig,
)

print('Registered backends:', sorted(BACKENDS))


## Pattern 1 — discriminated union

Every backend's config has `kind: Literal["..."] = "..."`. The union is
`Annotated[TFMLPConfig | TorchMLPConfig | ... , Field(discriminator='kind')]`.

Pydantic uses `kind` to pick the right subclass during validation — no isinstance checks, no manual dispatch.


In [ ]:
from pydantic import TypeAdapter
from deepCab.schemas.config import BackendConfig

ta = TypeAdapter(BackendConfig)

for cfg in [TFMLPConfig(), TorchMLPConfig(), XGBConfig(),
            LGBMConfig(), CatBoostConfig(), FTTransformerConfig()]:
    blob = cfg.model_dump()
    rebuilt = ta.validate_python(blob)
    assert rebuilt.kind == cfg.kind
    print(f'  {cfg.kind:18s} round-trips ✓')


## Pattern 2 — plain-dict registry

`BACKENDS: dict[str, type[AbstractEstimator]]` — keyed by `cfg.kind`, no decorator magic.
Adding a backend = appending one line. Easy to grep, easy to extend.


In [ ]:
import numpy as np

def _avail(mod):
    try: __import__(mod); return True
    except: return False

# Build + fit + predict for whichever backends actually import locally.
X = np.random.default_rng(0).normal(size=(64, 6)).astype('float32')
y = X.sum(axis=1)

cases = [
    (LGBMConfig(n_estimators=10, num_leaves=4), 'lightgbm'),
    (XGBConfig(n_estimators=10, max_depth=3), 'xgboost'),
    (CatBoostConfig(iterations=10, depth=3), 'catboost'),
]
for cfg, dep in cases:
    if not _avail(dep):
        print(f'  {cfg.kind:18s} skipped ({dep} not installed)'); continue
    est = build_estimator(cfg)
    est.fit(X, y)
    pred = est.predict(X[:4])
    print(f'  {cfg.kind:18s} predict[:4] = {pred.round(3).tolist()}')


## Pattern 3 — sklearn compatibility

Every backend is `BaseEstimator + RegressorMixin`. So you get `cross_validate`, `GridSearchCV`, `Pipeline` — for free. The cross-validation harness in `training/cv.py` uses the spec-as-factory pattern (rebuild from `cfg` each fold) instead of `sklearn.clone()` — `clone` fails silently on framework mixes (TF / Torch state).


In [ ]:
from sklearn.model_selection import cross_val_score

if _avail('lightgbm'):
    est = build_estimator(LGBMConfig(n_estimators=10, num_leaves=4))
    scores = cross_val_score(est, X, y, cv=3, scoring='neg_mean_absolute_error')
    print(f'LGBM 3-fold MAE: {-scores}')


## What's next

- `03-hydra-optuna-cv.ipynb` — same union, driven by Hydra + Optuna with time-series CV.
- `06-onnx-serving.ipynb` — `to_onnx()` per backend; the registry powers `/predict`.
- `CONTRIBUTING.md → Adding a new backend` — 6 steps to wire a new one.
